# 03 Modelling

Baselines first, then the models, all inside the same temporal split. The fitting
code is `src/model.py`; this notebook shows the results and the reasoning around them.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, duckdb
pd.set_option('display.width', 160)

In [ ]:
from src.features import build, NUMERIC, CATEGORICAL, TARGET
from src.model import temporal_folds
df = build(pd.read_parquet('../data/processed/projetos.parquet'))
s = df[df.in_model_sample]
s.groupby(['split', 'ano_projeto']).size().unstack(fill_value=0)

## Proponent history is strictly backward looking

2019 has no history because nothing precedes it in this data. If this table showed
history for 2019, the feature would be leaking the future into the past.

In [ ]:
s.groupby('ano_projeto').agg(n=('PRONAC','size'),
    with_history=('prior_projetos', lambda x: int(x.notna().sum())),
    pct=('prior_projetos', lambda x: round(x.notna().mean()*100, 1)))

## Results

Cross validated inside the training years, then the held out cohort scored once.

In [ ]:
pd.read_csv('../reports/cv_results.csv')

In [ ]:
pd.read_csv('../reports/test_results.csv', index_col=0)

Read the Brier column. The model gains PR-AUC over the heuristic and gives up
calibration, which is the wrong trade for a tool that displays a probability.
See `docs/model_card.md`.